# Figure 5.1: Soft-Label Voxel Signature (Load Pre-computed Data)

## 修正说明
本代码严格遵循“读取已处理数据”的原则：
- **Panel A**: 读取原始 High-Res `.mat` 数据。
- **Panel B**: 对 High-Res ROI 进行高斯平滑可视化 (用于展示 Partial Volume 物理原理)。
- **Panel C**: **直接读取** 硬盘上 `downsampling/3d` 文件夹中的下采样成品数据 (不再重新计算)。
- **Panel D**: 从 Panel C 的真实数据中提取 Voxel Bar Plot。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.gridspec as gridspec
import scipy.io as sio
import h5py
import pandas as pd
import scipy.ndimage as ndimage
import warnings

# 绘图风格设置
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'DejaVu Sans'],
    'font.size': 12,
    'axes.linewidth': 1.5,
    'svg.fonttype': 'none',
    'figure.dpi': 300
})
warnings.filterwarnings('ignore')

## 1. 路径配置 (请确认文件名)
请特别注意 `PATH_LOW_RES` 的具体文件名。通常 pipeline 输出的是 `.h5` 文件。

In [ ]:
# === 绝对路径配置 ===

# 1. 原始高分辨率数据 (High-Res Source)
PATH_HIGH_RES = "/home/jovyan/gpu_space/workspace_jiayi/alex_datasets/3D_minimal/ODP_01_qhlazec_3d_validated_minimal.mat"

# 2. 已下采样的成品数据 (Low-Res Target)
# 注意：请确认您的 downsampling/3d 文件夹下的具体文件名。通常包含 subject ID。
# 假设文件名保持 ID 结构，后缀可能为 .h5
PATH_LOW_RES = "/home/jovyan/gpu_space/workspace_jiayi/alex_datasets/downsampling/3d/ODP_01_qhlazec_3d_validated_minimal_downsampled.npz"
# 如果文件名不同（例如没有_minimal），请在此修改 -> 
# PATH_LOW_RES = "/home/jovyan/gpu_space/workspace_jiayi/alex_datasets/downsampling/3d/ODP_01_qhlazec_downsampled.npz"

# 3. 颜色表
PATH_LUT = "/home/jovyan/gpu_space/workspace_jiayi/KAN-git/KAN-Brain-Single-Voxel-Segmentaion/3D_dev/training/downsampling/Freesurfer_LUT_alex_labels_jiayi.xlsx"

# === 物理参数 (用于坐标对齐) ===
RES_HIGH = 0.65        # mm
RES_LOW_XY = 1.8       # mm
RES_LOW_Z = 3.0        # mm

# 坐标转换因子
SCALE_XY = RES_HIGH / RES_LOW_XY  # ~0.36
SCALE_Z = RES_HIGH / RES_LOW_Z    # ~0.21

# === ROI 选择 ===
# High-Res 空间下的坐标
SLICE_IDX_HIGH = 160        
CROP_CENTER_HIGH = (150, 200) # (Row, Col)
CROP_SIZE_HIGH = 80

## 2. 数据加载函数
分别处理 `.mat` (原始) 和 `.h5` (下采样后)。

In [ ]:
def load_lut(lut_path):
    """加载 LUT"""
    df = pd.read_excel(lut_path)
    lut_dict = {}
    for _, row in df.iterrows():
        try:
            idx = int(row['ID']) if 'ID' in row else int(row.iloc[0])
            r = row['R'] if 'R' in row else row.iloc[2]
            g = row['G'] if 'G' in row else row.iloc[3]
            b = row['B'] if 'B' in row else row.iloc[4]
            name = row['LabelName'] if 'LabelName' in row else row.iloc[1]
            lut_dict[idx] = {'color': np.array([r, g, b]) / 255.0, 'name': name}
        except:
            continue
    return lut_dict

def load_high_res_data(path):
    """加载原始 High-Res .mat"""
    print(f"Loading High-Res: {path}")
    mat = sio.loadmat(path)
    labels = mat['region_labels']
    if 'data' in mat:
        anatomy = mat['data'][..., 0] # 假设通道0是解剖像
    else:
        anatomy = np.ones_like(labels, dtype=float)
    return anatomy, labels

def load_low_res_data(path):
    """加载已生成的 Low-Res .h5"""
    print(f"Loading Low-Res: {path}")
    # 根据您的 pipeline，输出通常是 h5
    # 结构通常包含 'soft_labels' 或 'region_labels' (soft)
    with h5py.File(path, 'r') as f:
        # 打印一下 keys 方便调试，如果不确定结构
        # print("Keys in H5:", list(f.keys()))
        
        # 读取 Soft Labels (Probability map)
        # 假设 key 是 'soft_labels' 或 'region_labels' 且维度是 (H, W, D, C) 或 (C, H, W, D)
        if 'soft_labels' in f:
            data = f['soft_labels'][:]
        elif 'region_labels' in f:
            data = f['region_labels'][:]
        else:
            raise KeyError(f"Could not find label dataset in {path}. Keys: {list(f.keys())}")
        
        # 确保维度顺序是 (H, W, D, C)
        # 有些 pipeline 可能是 (C, D, H, W), 需要检查
        # 这里假设是标准的 spatial first: (H, W, D, C)
        # 如果是 (D, H, W, C)，请根据实际情况调整 transpose
        
    return data

# === 执行加载 ===
lut = load_lut(PATH_LUT)
high_anat, high_labels = load_high_res_data(PATH_HIGH_RES)
low_soft_labels = load_low_res_data(PATH_LOW_RES)

print(f"High Res Shape: {high_labels.shape}")
print(f"Low Res Shape: {low_soft_labels.shape}")

## 3. 对齐与提取 (Alignment & Extraction)
因为 Low-Res 是直接读取的，我们需要计算 High-Res ROI 对应的 Low-Res 索引。

In [ ]:
# 1. High-Res ROI 提取
sl_h = slice(CROP_CENTER_HIGH[0]-CROP_SIZE_HIGH//2, CROP_CENTER_HIGH[0]+CROP_SIZE_HIGH//2)
roi_high_anat = high_anat[sl_h, sl_h, SLICE_IDX_HIGH]
roi_high_labels = high_labels[sl_h, sl_h, SLICE_IDX_HIGH]

# 2. Panel B 模拟生成 (仅用于可视化)
# 这里的平滑仅为了展示原理，不用于数据流
# 将 ROI 转为 One-hot
u_labels = np.unique(roi_high_labels)
u_labels = u_labels[u_labels != 0]
one_hot_viz = np.zeros((*roi_high_labels.shape, len(u_labels)))
for i, l in enumerate(u_labels):
    one_hot_viz[..., i] = (roi_high_labels == l).astype(float)
# 计算各向异性 Sigma (High Res Pixels)
sigma_pix = (RES_LOW_XY / 2.355) / RES_HIGH
smoothed_viz = ndimage.gaussian_filter(one_hot_viz, sigma=(sigma_pix, sigma_pix, 0))
# Normalize
smoothed_viz /= (np.sum(smoothed_viz, axis=-1, keepdims=True) + 1e-9)

# 3. Low-Res ROI 提取 (从读取的数据中)
# 计算 Low-Res 下的对应坐标
slice_idx_low = int(SLICE_IDX_HIGH * SCALE_Z)
center_row_low = int(CROP_CENTER_HIGH[0] * SCALE_XY)
center_col_low = int(CROP_CENTER_HIGH[1] * SCALE_XY)
size_low = int(CROP_SIZE_HIGH * SCALE_XY)

sl_l = slice(center_row_low - size_low//2, center_row_low + size_low//2)

# 提取 Low Res 切片 (注意：Low Res 数据通常是 (H, W, D, C))
# 需要处理一下边界，防止切出界
try:
    roi_low_soft = low_soft_labels[sl_l, sl_l, slice_idx_low, :]
except IndexError:
    print("Warning: Low Res coordinates out of bounds. Using center crop for demo.")
    # Fallback logic if mapping is slightly off
    H, W, D, C = low_soft_labels.shape
    roi_low_soft = low_soft_labels[H//2-10:H//2+10, W//2-10:W//2+10, slice_idx_low, :]

# 找出 ROI 中活跃的类别 (为了颜色映射)
# 注意：Low Res 包含 102 个通道，我们只显示 Top N 活跃的
mean_probs = np.mean(roi_low_soft, axis=(0,1))
active_indices = np.argsort(mean_probs)[::-1][:5] # 取该区域最活跃的5个Label
active_labels = active_indices # 这里 indices 直接对应 label ID (假设 channel index = label ID)
# 如果 Channel Index 不等于 Label ID (例如 label list 映射)，需额外处理
# 假设 Alexander 的 pipeline 输出是 dense 102 channels matching LUT ID directly?
# 通常是: channel 0 -> label 0, channel 1 -> label 1...

print(f"Mapped High Slice {SLICE_IDX_HIGH} -> Low Slice {slice_idx_low}")
print(f"Low Res ROI Shape: {roi_low_soft.shape}")

In [ ]:
# === 绘图函数 (保持不变) ===
def map_labels_to_rgb(label_map, lut, alpha=0.6):
    h, w = label_map.shape
    rgba = np.zeros((h, w, 4))
    for lab in np.unique(label_map):
        if lab == 0: continue
        c = lut.get(lab, {'color': [1,0,0]})['color']
        rgba[label_map == lab, :3] = c
        rgba[label_map == lab, 3] = alpha
    return rgba

def map_soft_tensor_to_rgb(soft_tensor, lut):
    """将 (H, W, C) 概率张量映射为 RGB"""
    h, w, n_classes = soft_tensor.shape
    rgb = np.zeros((h, w, 3))
    # 遍历所有通道
    for c in range(n_classes):
        prob_map = soft_tensor[..., c]
        if np.max(prob_map) < 0.01: continue # 忽略微小贡献
        
        color = np.array(lut.get(c, {'color': [0,0,0]})['color'])
        # Accumulate: Prob * Color
        for k in range(3):
            rgb[..., k] += prob_map * color[k]
            
    return np.clip(rgb, 0, 1)

# === 绘制 Figure 5.1 ===
fig = plt.figure(figsize=(18, 6), constrained_layout=True)
gs = gridspec.GridSpec(1, 4, width_ratios=[1, 1, 1, 0.6], figure=fig)

ax1 = fig.add_subplot(gs[0])
ax2 = fig.add_subplot(gs[1])
ax3 = fig.add_subplot(gs[2])
ax4 = fig.add_subplot(gs[3])

# Panel A: High-Res Hard
ax1.imshow(roi_high_anat, cmap='gray', extent=[0, CROP_SIZE_HIGH, CROP_SIZE_HIGH, 0])
ax1.imshow(map_labels_to_rgb(roi_high_labels, lut), interpolation='none', extent=[0, CROP_SIZE_HIGH, CROP_SIZE_HIGH, 0])
ax1.set_title("A. Original High-Res\n(MPRAGE Hard Labels)", fontweight='bold')
ax1.axis('off')

# Panel B: Visual Smoothing (Simulated for Demo)
ax2.imshow(roi_high_anat, cmap='gray', extent=[0, CROP_SIZE_HIGH, CROP_SIZE_HIGH, 0])
# 这里的 smoothed_viz 是基于 active classes 做的可视化
# 为了显示正确颜色，我们需要把 viz 的 index 映射回真实 label ID
# 上面生成 smoothed_viz 时用的 u_labels 是真实 ID
rgb_smooth = np.zeros((*roi_high_labels.shape, 3))
for i, label_id in enumerate(u_labels):
    prob = smoothed_viz[..., i]
    col = np.array(lut.get(label_id, {'color': [1,1,1]})['color'])
    rgb_smooth += prob[..., None] * col
ax2.imshow(np.clip(rgb_smooth, 0, 1), alpha=0.8)
ax2.set_title("B. Physical Smoothing\n(Simulating Partial Volume)", fontweight='bold')
ax2.axis('off')

# Panel C: Actual Low-Res Data (Loaded from Disk)
# 使用 map_soft_tensor_to_rgb 处理真实的 102 通道数据
rgb_low = map_soft_tensor_to_rgb(roi_low_soft, lut)
im3 = ax3.imshow(rgb_low, interpolation='nearest') # Nearest 才能显示格子
ax3.set_title("C. Native CEST Soft Targets\n(Loaded from Dataset)", fontweight='bold')

# 绘制 Grid
ax3.set_xticks(np.arange(-0.5, roi_low_soft.shape[1], 1), minor=True)
ax3.set_yticks(np.arange(-0.5, roi_low_soft.shape[0], 1), minor=True)
ax3.grid(which='minor', color='white', linestyle='-', linewidth=0.5, alpha=0.5)
ax3.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)

# Panel D: Bar Plot
# 找一个混合度高的像素
entropy = -np.sum(roi_low_soft * np.log(roi_low_soft + 1e-9), axis=-1)
py, px = np.unravel_index(np.argmax(entropy), entropy.shape)

# 在 Panel C 标记
rect = patches.Rectangle((px-0.5, py-0.5), 1, 1, linewidth=2, edgecolor='cyan', facecolor='none')
ax3.add_patch(rect)

# 画 Bar
voxel_probs = roi_low_soft[py, px, :]
top_k = np.argsort(voxel_probs)[::-1][:4]
names = [lut.get(idx, {'name': str(idx)})['name'] for idx in top_k]
colors = [lut.get(idx, {'color': [0.5,0.5,0.5]})['color'] for idx in top_k]
vals = voxel_probs[top_k]

bars = ax4.bar(range(4), vals, color=colors, edgecolor='k')
ax4.set_xticks(range(4))
ax4.set_xticklabels(names, rotation=45, ha='right')
ax4.set_ylim(0, 1.0)
ax4.set_title("Soft Voxel Signature")
ax4.spines['right'].set_visible(False)
ax4.spines['top'].set_visible(False)
for b in bars:
    ax4.text(b.get_x()+b.get_width()/2, b.get_height(), f"{b.get_height():.2f}", ha='center', va='bottom')

plt.suptitle(f"Figure 5.1: Soft-Label Generation (Source: {PATH_LOW_RES.split('/')[-1]})", y=1.05)
plt.show()